## Exploring EPMT DB

This notebook is trying to explore beyond the 'EPMT_JOB_TAGS' within 'annotations' for rows within the EPMT database.

In [ ]:
# Environment work around to get matplotlib/seaborn
import sys
sys.path.append('/home/Janice.Kim/work/epmt/misc/jk_py37')

In [ ]:
import epmt_query as eq
import orm.sqlalchemy.models as models
from pprint import pprint
from datetime import datetime
from pathlib import Path
import csv
import pandas as pd

In [ ]:
# Query how many rows exist
eq.get_jobs(fmt='orm').count()

In [ ]:
one=eq.get_jobs(fmt='dict', limit=1)[0]
pprint(one)

In [ ]:
# These look potentially useful...
print(one['env_dict']['SLURM_MEM_PER_CPU'])
print(one['env_dict']['SLURM_MEM_PER_NODE'])

In [ ]:
jobs=eq.get_jobs(fmt='dict', limit=100)

In [ ]:
jobs[:]
mem_per_cpu = set()
mem_per_node = set()
loaded_modules = set()
mem_per_node_missing_count = 0
jobids = set()

for job in jobs:
    jobids.add(job['jobid'])
    mem_per_cpu.add(job['env_dict']['SLURM_MEM_PER_CPU'])
    if 'SLURM_MEM_PER_NODE' in job['env_dict']:
        mem_per_node.add(job['env_dict']['SLURM_MEM_PER_NODE'])
    else:
        mem_per_node_missing_count += 1
    loaded_modules.add(job['env_dict']['LOADEDMODULES'])
print(mem_per_cpu, mem_per_node, loaded_modules, mem_per_node_missing_count)
print(len(jobids)) # Should be 100, so likely this is the primary key. Checked postgres db on jobs table, and it is.

## More about the data

See the following link for more description about the data: https://gitlab.com/minimal-metrics-llc/epmt/epmt#performance-metrics-data-dictionary

There are more features to explore or predict. For example:
* 41. read_bytes Thread Bytes read from I/O device 
* 42. write_bytes Thread Bytes written to I/O device
* 45. time_waiting Thread Nanoseconds runnable but waiting


In [ ]:
# Just some that look potentially interesting...
print(one['read_bytes'])
print(one['write_bytes'])
print(one['time_waiting'])
print(one['cpu_time'])
print(one['duration'])
print(one['start'])
print(one['end'])
print(one['minflt'])
print(one['majflt'])
print(one['tags'])
print(one['annotations'])
print(one['env_dict']['SLURM_JOB_ACCOUNT'])
print(one['env_dict']['SLURM_NTASKS'])
print(one['env_dict']['SLURM_TASKS_PER_NODE'])
print(one['env_dict']['SLURM_MEM_PER_CPU'])
print(one['env_dict']['SLURM_MEM_PER_NODE'])
print(one['env_dict']['SLURM_SCRIPT_CONTEXT'])
print(one['env_dict']['LOADEDMODULES'])
print(one['all_proc_tags']) # Maybe features?
print(one['exitcode']) # Maybe prune out the ones that aren't 0?

## Let's collect all of these for 40-50k rows

I'm not sure the best way to do this. I'll try just to save the data as is into a csv file. Once we have it, we can load it into memory and try to clean it up later.

In [ ]:
# Actually no... let's not do this. It seems to thrash when I go from 40k to 50k, so I'll comment this part out.
'''
nrows = 10000 # TODO: increase limit once things look okay, but 50k may be too large at one time? 40k finishes...
all_jobs = eq.get_jobs(fmt='dict', limit=nrows) 

keys = ['read_bytes', 'write_bytes', 'time_waiting','cpu_time', 'duration', 'start', 'end', 'minflt', 'majflt', 'tags', 'annotations', 'all_proc_tags', 'exitcode']
env_dict_keys = ['SLURM_JOB_ACCOUNT', 'SLURM_NTASKS', 'SLURM_TASKS_PER_NODE', 'SLURM_MEM_PER_CPU', 'SLURM_MEM_PER_NODE', 'SLURM_SCRIPT_CONTEXT', 'LOADEDMODULES']

data = []
for job in all_jobs:
    row = {}
    for key in keys:
        if key in job:
            row[key] = job[key]
        else:
            row[key] = ""
    for ed_key in env_dict_keys:
        if ed_key in job['env_dict']:
            row[ed_key] = job['env_dict'][ed_key]
        else:
            row[ed_key] = ""
    data.append(row)

labels = data[0].keys()
'''

In [ ]:
'''
# I don't want a massive csv file, so I'm going to chunk it for now. I like csv files
# because I can just open them up and look at them if I want. In the future, maybe 
# something like parquet or feather are better.

total_rows = len(data)
print(f"Processing {total_rows} rows.")

max_rows_per_file = 5000
date_str = datetime.today().strftime("%Y%m%d_%H%M%S")

work_dir = Path(f"/home/Janice.Kim/work/epmt/data_{total_rows}_{date_str}")
work_dir.mkdir(parents=True, exist_ok=True)

for i in range(0, total_rows, max_rows_per_file):
    data_chunk = data[i:i+max_rows_per_file]
    part_num = i // max_rows_per_file + 1
    filepath = work_dir / f"jk_data_{total_rows}_{part_num}_{date_str}.csv"
    print(filepath)

    with open(filepath, "w", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=labels)
        writer.writeheader()
        writer.writerows(data_chunk)
'''

## Actually, Let's Query the DB in Chunks
Instead of getting 50k rows all in one go, let's query the db in chunks.... Let's query the db in 5k chunks and write those chunks to file. Let's use the jobid to help us order the results and make sure we don't miss or duplicate anything...

In [ ]:
# How do I get the min_id and max_id?
# WIP
from sqlalchemy import func
#tags = ['jobid', 'env_dict', 'read_bytes', 'write_bytes', 'time_waiting','cpu_time', 'duration', 'start', 'end', 'minflt', 'majflt', 'tags', 'annotations', 'all_proc_tags', 'exitcode']
tags = ['jobid']

all_jobs = eq.get_jobs(
    fmt='orm',
    order=models.Job.jobid.asc(),
)
print(type(all_jobs))
print('Total job count: ', all_jobs.count())

In [ ]:
# Sigh... there's probably a better way to do this
min_jobid = all_jobs[0].jobid
max_jobid = all_jobs[-1].jobid
print('Min/Max jobid = ', min_jobid, max_jobid)
# Min/Max jobid =  1011 52546867

In [ ]:
#WIP - not working yet....
'''
total_row_count = 10 # Ideally, we want this to be: eq.get_jobs(fmt='orm').count()
date_str = datetime.today().strftime("%Y%m%d_%H%M%S")
work_dir = Path(f"/home/Janice.Kim/work/epmt/data_{total_row_count}_{date_str}")
work_dir.mkdir(parents=True, exist_ok=True)

keys = ['jobid', 'env_dict', 'read_bytes', 'write_bytes', 'time_waiting','cpu_time', 'duration', 'start', 'end', 'minflt', 'majflt', 'tags', 'annotations', 'all_proc_tags', 'exitcode']

batch_size = 5
batch_idx = 0
last_id = None

while True:
    batch_jobs = eq.get_jobs(
    fmt='dict',
    order=models.Job.jobid.asc(), 
    fltr=(models.Job.jobid <= max_jobid), 
    limit=batch_size)
    
    if last_id is not None:
        batch_jobs = batch_jobs.filter(models.Job.jobid > last_id)

    rows = batch_jobs
    total_rows = len(rows)
    print(f"Processing {total_rows} rows.")

    filepath = work_dir / f"jk_data_{batch_idx}_{date_str}.csv"
    print(filepath)

    labels = rows[0].keys()
    
    with open(filepath, "w", encoding="utf-8") as csv_file:
        writer = csv.DictWriter(csv_file, fieldnames=labels)
        writer.writeheader()
        writer.writerows(rows)

    break # FOR NOW
    
    last_id = rows[-1].jobid
'''

In [ ]:
# On hold for now... I have 40k rows of data. I'll come back to this if I need to pull data later.

Now that we have it in a file, can we clean it up and make it useful? - TODO in another notebook